In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="mistral-small-latest",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="mistral-small-latest",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nAnswer a series of questions about the fictional moon city of Lunapolis, including its capital, weather, cheese miners, and their union's potential strike.\n\n## SUMMARY\nLunapolis is introduced as the capital of the moon. Its weather is extreme: clear skies with temperatures ranging from 120°C high to -100°C low. Lunapolis has a population of 100,000 cheese miners, who are unhappy with the new president and may go on strike.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e065a65d-f21c-4a38-92e5-70e3aa4b307f'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='8d1a5b1d-7bce-48d5-ae4d-da7e86df43d4'),
              AIMessage(content='As the newly installed president of Lunapolis, I\'d dep

In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
Answer a series of questions about the fictional moon city of Lunapolis, including its capital, weather, cheese miners, and their union's potential strike.

## SUMMARY
Lunapolis is introduced as the capital of the moon. Its weather is extreme: clear skies with temperatures ranging from 120°C high to -100°C low. Lunapolis has a population of 100,000 cheese miners, who are unhappy with the new president and may go on strike.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [7]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [8]:
agent = create_agent(
    model="mistral-small-latest",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='8da8d345-ee3e-4e43-a92f-27646438e3cf'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='dbd8a5de-f009-4696-8131-a789600c21f7', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='f2d55d88-49b9-4c50-a231-0d45e07a6f84'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='ecf91fb6-37da-4897-872f-0311a9040d96', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='802ff813-9ecf-48fd-8e07-0870342a8a06'),
              AIMessage(content="I can't check the device's temperature directly, but if it's o

In [10]:
print(response["messages"][-1].content)

I can't check the device's temperature directly, but if it's overheating, it might shut down or behave strangely.

Here are a few things you can try to check its temperature safely:

1. **Touch Test (Carefully)** – If you can feel the device (like a phone, laptop, or tablet), check if it's unusually hot. If it feels too hot to touch for more than a few seconds, it might be overheating.

2. **Look for Signs** – Some devices have overheating warnings (like error messages or thermal throttling).

3. **Check Air Vents** – If it has vents, make sure they're not blocked.

If it's overheating, try:
- Turning it off and letting it cool down.
- Moving it to a cooler place.
- Checking for dust buildup in vents (if it's a computer or similar device).
